<a href="https://colab.research.google.com/github/ja2faded/Machine-Learning-Final-Project/blob/main/MachineLearningProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Among MLB teams, can team stats through a team's first 40 games (OBP, SLG, ERA, run differential) predict final season win totals more accurately than win percentage alone (through first 40 games)?

**Team members:** Derek, Will, Jackson, Abdullah
**Repository:** https://github.com/ja2faded/Machine-Learning-Final-Project

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [119]:
from pathlib import Path

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

## Load data

In [120]:
# Load data from DATA_DIR.
import pandas as pd

gi = pd.read_csv(DATA_DIR / 'gameinfo.csv', low_memory=False)

ts = pd.read_csv(DATA_DIR / 'teamstats.csv', low_memory=False)

print(gi.shape)
print(ts.shape)




(226221, 43)
(506130, 111)


## Prepare data

In [121]:
# Clean data and create features.
#keep regular season games from 1940-2025 (earlier games have more missing values)

gi_f = gi[(gi['season'] >= 1940) & (gi['season'] <= 2025) & (gi['gametype'] == 'regular')].copy()
gid_set = set(gi_f['gid'])

#keep only rows for the games we kept above and 'value'
ts_f = ts[ts['gid'].isin(gid_set) & (ts['stattype'] == 'value')].drop(columns=['stattype']).copy()

#make strings numeric columns
num_cols = ['b_h', 'b_d', 'b_t', 'b_hr', 'b_ab', 'b_w', 'b_hbp', 'b_sf', 'p_er', 'p_ipouts', 'b_r', 'p_r', 'win']
for col in num_cols:
    ts_f[col] = pd.to_numeric(ts_f[col], errors='coerce')

#attach seasons to teamstats
#correctly format date
ts_f = ts_f.merge(gi_f[['gid', 'season']], on= 'gid', how = 'left')
ts_f['game_date'] = pd.to_datetime(ts_f['date'], format = '%Y%m%d')
ts_f = ts_f.sort_values(['team', 'season', 'game_date'])
ts_f['game_num'] = ts_f.groupby(['team', 'season']).cumcount() + 1

#first 40 games
first40 = ts_f[ts_f['game_num'] <= 40].copy()


#sum first 40 game stats per team-season
sums40 = first40.groupby(['team', 'season'])[
    ['b_h', 'b_d', 'b_t', 'b_hr', 'b_ab', 'b_w', 'b_hbp', 'b_sf', 'p_er', 'p_ipouts', 'b_r', 'p_r']
    ].sum().reset_index()


#win percentage through first 40
winpct40 = first40.groupby(['team', 'season'])['win'].mean().reset_index(name='first40_win_pct')
agg40 = sums40.merge(winpct40, on = ['team', 'season'], how = 'left')

#compute OBP, SLG, ERA, run differential
agg40['OBP'] = (agg40['b_h'] + agg40['b_w'] + agg40['b_hbp']) / \
                (agg40['b_ab'] + agg40['b_w'] + agg40['b_hbp'] + agg40['b_sf'])

agg40['SLG'] = (agg40['b_h'] + agg40['b_d'] + 2*agg40['b_t'] + 3*agg40['b_hr']) / \
                (agg40['b_ab'])

agg40['ERA'] = agg40['p_er'] *27 / agg40['p_ipouts']

agg40['RD'] = agg40['b_r'] - agg40['p_r']

#full season win total
season_wins = ts_f.groupby(['team', 'season'])['win'].agg(games_played = 'count', wins = 'sum').reset_index()

#merge features and target
final = agg40[['team', 'season', 'OBP', 'SLG', 'ERA', 'RD', 'first40_win_pct']].merge(
        season_wins, on = ['team', 'season'], how = 'left')

# Full season win percentage
# Only include MLB seasons of adequate length (excludes COVID, lockouts, etc.)
final = final[final['games_played'] >= 100].copy()
final['final_win_pct'] = final['wins']/final['games_played']

print(final.shape)
print(final.head())
print()
print(final.isna().sum())




(2052, 10)
  team  season       OBP       SLG       ERA    RD  first40_win_pct  \
0  ANA    1997  0.339514  0.398155  4.716266  -4.0            0.525   
1  ANA    1998  0.337049  0.433700  4.684701  -3.0            0.525   
2  ANA    1999  0.326544  0.426396  4.448571   3.0            0.450   
3  ANA    2000  0.359205  0.486272  5.394958 -11.0            0.475   
4  ANA    2001  0.327152  0.399853  4.007612   4.0            0.475   

   games_played  wins  final_win_pct  
0           162    84       0.518519  
1           162    85       0.524691  
2           162    70       0.432099  
3           162    82       0.506173  
4           162    75       0.462963  

team               0
season             0
OBP                0
SLG                0
ERA                0
RD                 0
first40_win_pct    0
games_played       0
wins               0
final_win_pct      0
dtype: int64


## Explore data

In [122]:
# Summarize and visualize the prepared data.
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np




## Build models

In [123]:
# Train baseline and candidate models.

## Evaluate models

In [124]:
# Compare models using the chosen evaluation measures.

## Final results

In [125]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]